In [50]:
import os
import sys
sys.stdout.flush()
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, AIMessageChunk
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

In [51]:
load_dotenv()

True

In [52]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1",
    temperature=0
)

In [53]:
class State(TypedDict):
    message: Annotated[list, add_messages]

SYSTEM_PROMPT = SystemMessage(content="You are a helpful assistant.")    

In [54]:
def chat_node(state: State):
    response = llm.invoke([SYSTEM_PROMPT] + state["message"])
    return {"message": [response]}

In [55]:
graph = StateGraph(State)
checkpointer = MemorySaver()

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

workflow = graph.compile(checkpointer=checkpointer)


In [56]:
thread_id = "1"
while True:

     user_input = input("You") 

     print("User:",user_input)

     if user_input.strip().lower() in ["exit","quit","bye"]:
      break

     config = {'configurable':{'thread_id':thread_id}}

     response = workflow.invoke({"message": [HumanMessage(content=user_input)]},config=config)
     print("AI:",response['message'][-1].content)




User: hello my name is vishal
AI: Hello Vishal! Nice to meet you. How can I help you today?
User: what is my name
AI: Your name is **Vishal** — you mentioned it earlier! 😊
User: what is 100 + 100
AI: 100 + 100 is **200**. 😊
User: now multiply the result with 2
AI: 400
User: exit
